# Build the validation deck — "Validation of Predicted Imperviousness - deck.pptx"

Packaged from `make_validation_deck.py`, a single self-contained script: an
11-slide deck built directly from the three cross-city validation-technique
CSVs, companion to the Word validation report.

In [ ]:
# -*- coding: utf-8 -*-
"""Build the ground-truth validation presentation.

The slide counterpart of the "Validation of Predicted Imperviousness" report:
twenty predicted-imperviousness rasters checked against each city's 450-plot
photo-interpreted ground truth with three complementary techniques. Every number
is read at build time from the confusion-matrix validation CSVs, so the deck
regenerates cleanly whenever the validation is re-run.

    C:\\ProgramData\\anaconda3\\python.exe make_validation_deck.py

Writes the .pptx next to the report and its figures under figs_deck/.
"""
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Emu, Inches, Pt

# --------------------------------------------------------------------- paths
CSV = (r"C:\Users\user\OneDrive - Politecnico di Milano\File di Daniele Oxoli - PhD_Keerthana"
       r"\test_embeddings\Groundtruth_validation\output\confusion_matrix_validation\cross_city")
OUT_DIR = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
           r"\Ground_truth_validation_S2\Presentation")
OUT_PPTX = os.path.join(OUT_DIR, "Validation of Predicted Imperviousness - deck.pptx")
FIGS = "figs_deck"
os.makedirs(FIGS, exist_ok=True)

# --------------------------------------------------------------------- style
INK, MUTED, RULE = "#1a1a1a", "#6b6b6b", "#d4d4d4"
C_CLMS_MODEL = "#0B6E4F"   # CLMS-trained models
C_GHSL_MODEL = "#C2724A"   # GHSL-trained models
C_BENCH = "#7d93a3"        # benchmark products
W, H = Inches(13.333), Inches(7.5)

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.color": "#ececec", "grid.linewidth": 0.8,
    "axes.axisbelow": True,
})

# --------------------------------------------------------------------- data
A = pd.read_csv(os.path.join(CSV, "technique_A_continuous_metrics.csv")).set_index(["dataset", "raster"])
B = pd.read_csv(os.path.join(CSV, "technique_B_hard_confusion_metrics.csv")).set_index(["dataset", "raster"])
C = pd.read_csv(os.path.join(CSV, "technique_C_level_confusion_metrics.csv")).set_index(["dataset", "raster"])

MILAN = [("S2_stack", "clms"), ("S2_percentile", "clms"), ("S2_median", "clms"), ("emb_RF", "clms"),
         ("S2_stack_GHSL", "ghsl"), ("S2_percentile_GHSL", "ghsl"),
         ("S2_median_GHSL", "ghsl"), ("emb_GHSL", "ghsl"),
         ("CLMS", "bench"), ("GHSL", "bench")]
VN = [("emb_localrf", "ghsl"), ("s2_median_localrf", "ghsl"),
      ("s2_median_zeroshot", "ghsl"), ("emb_zeroshot", "ghsl"), ("GHSL", "bench")]
CITY = [("Milan_2018", "Milan", MILAN), ("Hanoi_2018", "Hanoi", VN), ("HCMC_2018", "HCMC", VN)]
COL = {"clms": C_CLMS_MODEL, "ghsl": C_GHSL_MODEL, "bench": C_BENCH}
KEY = [(C_CLMS_MODEL, "model, CLMS labels"),
       (C_GHSL_MODEL, "model, GHSL labels"),
       (C_BENCH, "benchmark product")]


def gv(df, ds, r, c):
    return float(df.loc[(ds, r), c])


def save(fig, name):
    p = os.path.join(FIGS, name)
    fig.savefig(p, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return p


# --------------------------------------------------------------------- figures
def fig_metric(df, col, lower_better, title, fname):
    fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.4))
    for ax, (ds, city, lst) in zip(axes, CITY):
        names = [r for r, _ in lst]
        vals = [gv(df, ds, r, col) for r in names]
        cols = [COL[g] for _, g in lst]
        y = np.arange(len(names))[::-1]
        ax.barh(y, vals, color=cols, height=0.68)
        ax.set_yticks(y)
        ax.set_yticklabels(names, fontsize=8)
        ax.set_title(city, fontsize=12, fontweight="bold")
        for yi, v in zip(y, vals):
            ax.text(v, yi, f" {v:.2f}", va="center", fontsize=7.5, color=MUTED)
        ax.margins(x=0.18)
    axes[0].set_xlabel(title, fontsize=10)
    hint = "lower is better" if lower_better else "higher is better"
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c, _ in KEY]
    fig.legend(handles, [t for _, t in KEY], loc="lower center", ncol=3,
               frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.08))
    fig.text(0.995, -0.02, hint, ha="right", fontsize=9, color=MUTED)
    fig.tight_layout()
    return save(fig, fname)


def fig_trainlabel(fname):
    groups = ["CLMS-trained\nmodels", "Same features,\nGHSL-trained", "Raw GHSL\nproduct"]
    clms_m = ["S2_stack", "S2_percentile", "S2_median", "emb_RF"]
    ghsl_m = ["S2_stack_GHSL", "S2_percentile_GHSL", "S2_median_GHSL", "emb_GHSL"]

    def band(df, col):
        return [np.mean([gv(df, "Milan_2018", r, col) for r in clms_m]),
                np.mean([gv(df, "Milan_2018", r, col) for r in ghsl_m]),
                gv(df, "Milan_2018", "GHSL", col)]

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9))
    for ax, (col, df, ttl, lo) in zip(axes, [
        ("kappa", B, "Cohen's kappa (Technique B)", False),
        ("quad_weighted_kappa", C, "QWK (Technique C)", False),
        ("bias_pp", A, "Bias, pp (Technique A)", None)]):
        vals = band(df, col)
        ax.bar(range(3), vals, color=[C_CLMS_MODEL, C_GHSL_MODEL, C_BENCH], width=0.62)
        ax.set_xticks(range(3))
        ax.set_xticklabels(groups, fontsize=8.5)
        ax.set_title(ttl, fontsize=11, fontweight="bold")
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.2f}", ha="center",
                    va="bottom" if v >= 0 else "top", fontsize=9)
        if col == "bias_pp":
            ax.axhline(0, color=MUTED, lw=0.8)
    fig.suptitle("")
    fig.tight_layout()
    return save(fig, fname)


# --------------------------------------------------------------------- pptx helpers
def add_slide(prs):
    s = prs.slides.add_slide(prs.slide_layouts[6])
    bg = s.background.fill
    bg.solid()
    bg.fore_color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    return s


def textbox(slide, x, y, w, h, text, size=12, bold=False, color=INK,
            align=PP_ALIGN.LEFT, italic=False, spacing=1.22):
    tb = slide.shapes.add_textbox(x, y, w, h)
    tf = tb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(text.split("\n")):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        p.line_spacing = spacing
        r = p.add_run()
        r.text = line
        r.font.size = Pt(size)
        r.font.bold = bold
        r.font.italic = italic
        r.font.color.rgb = RGBColor.from_string(color.lstrip("#").upper())
        r.font.name = "Calibri"
    return tb


def rule(slide, y, x=Inches(0.72), w=Inches(11.9), h=Emu(9525)):
    from pptx.enum.shapes import MSO_SHAPE
    sh = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, x, y, w, h)
    sh.fill.solid()
    sh.fill.fore_color.rgb = RGBColor.from_string(RULE.lstrip("#").upper())
    sh.line.fill.background()
    sh.shadow.inherit = False


def header(slide, kicker, title, sub=None):
    textbox(slide, Inches(0.72), Inches(0.30), Inches(11.9), Inches(0.3),
            kicker.upper(), size=10.5, bold=True, color=C_CLMS_MODEL)
    textbox(slide, Inches(0.72), Inches(0.60), Inches(11.9), Inches(0.9),
            title, size=24, bold=True)
    y = Inches(1.42)
    if sub:
        textbox(slide, Inches(0.72), y, Inches(11.9), Inches(0.7),
                sub, size=13, color=MUTED, spacing=1.3)
        y = Inches(2.06)
    rule(slide, y)
    return y


def add_picture(slide, path, top, max_h, left=Inches(0.72), max_w=Inches(11.9)):
    iw, ih = Image.open(path).size
    ar = iw / ih
    w = max_w
    h = Emu(int(w / ar))
    if h > max_h:
        h = max_h
        w = Emu(int(h * ar))
    x = left + Emu(int((max_w - w) / 2))
    slide.shapes.add_picture(path, x, top, width=w, height=h)


def table(slide, df, x, y, w, h, size=11, highlight_row=None):
    rows, cols = df.shape[0] + 1, df.shape[1]
    g = slide.shapes.add_table(rows, cols, x, y, w, h).table
    for j, c in enumerate(df.columns):
        cell = g.cell(0, j)
        cell.text = str(c)
        run = cell.text_frame.paragraphs[0].runs[0]
        run.font.size = Pt(size)
        run.font.bold = True
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        run.font.name = "Calibri"
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor.from_string(INK.lstrip("#").upper())
    for i in range(df.shape[0]):
        hl = highlight_row is not None and i == highlight_row
        for j in range(cols):
            cell = g.cell(i + 1, j)
            cell.text = str(df.iat[i, j])
            run = cell.text_frame.paragraphs[0].runs[0]
            run.font.size = Pt(size)
            run.font.bold = hl
            run.font.name = "Calibri"
            run.font.color.rgb = RGBColor.from_string(
                (C_CLMS_MODEL if hl else INK).lstrip("#").upper())
            cell.fill.solid()
            cell.fill.fore_color.rgb = (RGBColor(0xEC, 0xF4, 0xF0) if hl
                                        else RGBColor(0xFF, 0xFF, 0xFF) if i % 2 == 0
                                        else RGBColor(0xF7, 0xF7, 0xF7))
    return g


# --------------------------------------------------------------------- build
def build():
    f_rmse = fig_metric(A, "RMSE_pp", True, "RMSE (pp)", "rmse.png")
    f_kappa = fig_metric(B, "kappa", False, "Cohen's kappa", "kappa.png")
    f_qwk = fig_metric(C, "quad_weighted_kappa", False, "QWK", "qwk.png")
    f_train = fig_trainlabel("trainlabel.png")

    prs = Presentation()
    prs.slide_width, prs.slide_height = W, H

    # 1 Title
    s = add_slide(prs)
    textbox(s, Inches(0.72), Inches(2.4), Inches(11.9), Inches(1.3),
            "Validation of Predicted Imperviousness Maps", size=34, bold=True)
    textbox(s, Inches(0.72), Inches(3.7), Inches(11.9), Inches(0.9),
            "Milan, Hanoi and Ho Chi Minh City, 2018", size=17, color=MUTED)
    textbox(s, Inches(0.72), Inches(4.5), Inches(11.9), Inches(1.6),
            "Twenty predicted rasters checked against each city's 450-plot photo-interpreted "
            "ground truth,\nusing three complementary techniques: continuous error, a hard "
            "confusion matrix at 50 %,\nand a threshold-free fraction-level confusion matrix.",
            size=13, color=MUTED, spacing=1.35)

    # 2 Why three techniques
    s = add_slide(prs)
    header(s, "Method", "One metric can mislead; three complementary views cannot",
           "All three run against the same 450 plots per city, each a 10 x 10 m cell "
           "photo-interpreted as nine sub-cells.")
    textbox(s, Inches(0.9), Inches(2.5), Inches(11.5), Inches(4),
            "Technique A  -  Continuous error.  Predicted % against reference %, no threshold. "
            "RMSE, MAE and bias in percentage points.\n\n"
            "Technique B  -  Hard confusion matrix at 50 %.  Each plot is predominantly pervious "
            "or impervious; Cohen's kappa is the headline metric.\n\n"
            "Technique C  -  Fraction-level (10-class) confusion matrix.  Ordinal agreement with "
            "no threshold; quadratic-weighted kappa is the headline metric.",
            size=14, spacing=1.4)

    # 3 What was validated
    s = add_slide(prs)
    header(s, "Catalogue", "Twenty rasters: models, and the benchmark products they were trained toward",
           "Milan is the only city with both label sources, so it isolates the effect of the "
           "training label.")
    cat = pd.DataFrame([
        ["Milan", "S2_stack / S2_percentile / S2_median / emb_RF", "Models trained on CLMS labels"],
        ["Milan", "S2_stack_GHSL / S2_percentile_GHSL / S2_median_GHSL / emb_GHSL", "Same features, trained on GHSL labels"],
        ["Milan", "CLMS, GHSL", "Benchmark products, scored as maps under test"],
        ["Hanoi / HCMC", "emb_localrf / emb_zeroshot / s2_median_localrf / s2_median_zeroshot", "Local re-fit and zero-shot transfer (GHSL labels)"],
        ["Hanoi / HCMC", "GHSL", "Benchmark product"],
    ], columns=["City", "Rasters", "Role"])
    table(s, cat, Inches(0.72), Inches(2.5), Inches(11.9), Inches(3.0), size=11)

    # 4 Technique A
    s = add_slide(prs)
    header(s, "Technique A", "Continuous error: S2_stack leads in Milan; GHSL is worst everywhere",
           "The GHSL-trained models sit between the CLMS-trained models and the raw GHSL product.")
    add_picture(s, f_rmse, Inches(2.4), Inches(4.5))

    # 5 Technique B
    s = add_slide(prs)
    header(s, "Technique B", "Hard confusion at 50 %: the same ordering by Cohen's kappa",
           "CLMS-trained models on top, then GHSL-trained models, then the raw GHSL product.")
    add_picture(s, f_kappa, Inches(2.4), Inches(4.5))

    # 6 Technique C
    s = add_slide(prs)
    header(s, "Technique C", "Fraction-level agreement: QWK confirms the ordering",
           "s2_median_zeroshot posts the best QWK in the catalogue in HCMC (0.80), provisional "
           "on a thin composite.")
    add_picture(s, f_qwk, Inches(2.4), Inches(4.5))

    # 7 Same top raster
    s = add_slide(prs)
    header(s, "Synthesis", "All three techniques pick the same best raster in every city",
           "The complete ranking is identical in Hanoi and HCMC; in Milan the lower ranks "
           "reshuffle, but the three bands hold.")
    win = pd.DataFrame([
        ["Milan", "S2_stack", f"{gv(A,'Milan_2018','S2_stack','RMSE_pp'):.2f}",
         f"{gv(B,'Milan_2018','S2_stack','kappa'):.3f}", f"{gv(C,'Milan_2018','S2_stack','quad_weighted_kappa'):.3f}"],
        ["Hanoi", "emb_localrf", f"{gv(A,'Hanoi_2018','emb_localrf','RMSE_pp'):.2f}",
         f"{gv(B,'Hanoi_2018','emb_localrf','kappa'):.3f}", f"{gv(C,'Hanoi_2018','emb_localrf','quad_weighted_kappa'):.3f}"],
        ["HCMC", "s2_median_zeroshot", f"{gv(A,'HCMC_2018','s2_median_zeroshot','RMSE_pp'):.2f}",
         f"{gv(B,'HCMC_2018','s2_median_zeroshot','kappa'):.3f}", f"{gv(C,'HCMC_2018','s2_median_zeroshot','quad_weighted_kappa'):.3f}"],
    ], columns=["City", "Best raster", "RMSE (pp)", "kappa", "QWK"])
    table(s, win, Inches(1.5), Inches(2.7), Inches(10.3), Inches(2.0), size=13)

    # 8 Training label
    s = add_slide(prs)
    header(s, "Key finding", "The training label is a first-order driver of map accuracy",
           "Milan, same features, CLMS labels vs GHSL labels vs the raw GHSL product.")
    add_picture(s, f_train, Inches(2.3), Inches(3.7))
    textbox(s, Inches(0.9), Inches(6.1), Inches(11.5), Inches(1.1),
            "A GHSL-trained model inherits about half of the product's +19 pp under-prediction "
            "bias and loses roughly 0.16 to 0.20 of kappa. The difference is statistically "
            "significant for every feature set (paired Wilcoxon).",
            size=12.5, color=MUTED, spacing=1.3)

    # 9 GHSL is a proxy
    s = add_slide(prs)
    header(s, "Interpretation", "GHSL measures built-up surface, not imperviousness",
           "So the weak GHSL results are a semantic mismatch, not product error.")
    textbox(s, Inches(0.9), Inches(2.5), Inches(11.5), Inches(3.8),
            "CLMS and the photo ground truth both measure the sealed fraction of the surface: "
            "roads, pavements, car parks and yards as well as buildings.\n\n"
            "GHSL (GHS-BUILT-S) measures only the roofed built-up fraction. It omits every "
            "non-roof sealed surface by construction, and reads about 19 pp lower than a true "
            "imperviousness product across all three cities.\n\n"
            "GHSL is still the only globally consistent benchmark outside Europe, but its values "
            "should not be compared directly against an imperviousness reference without "
            "acknowledging that offset.",
            size=13.5, spacing=1.4)

    # 10 No percentile for Vietnam
    s = add_slide(prs)
    header(s, "Limitation", "The percentile representation cannot be built for Hanoi or HCMC",
           "It needs at least 17 cloud-free dates; the 2018 archive does not have them.")
    vn = pd.DataFrame([
        ["Hanoi", "10", "4", "30"],
        ["HCMC", "16", "3", "30"],
        ["Milan (reference)", "-", "30", "30"],
    ], columns=["City", "2018 L2A scenes", "Usable dates", "Milan"])
    table(s, vn, Inches(1.6), Inches(2.7), Inches(10.1), Inches(2.0), size=13)
    textbox(s, Inches(0.9), Inches(5.2), Inches(11.5), Inches(1.5),
            "The Vietnamese median composites are only about three clear observations per pixel "
            "(dry season, not annual), so the strong HCMC zero-shot result is confounded with "
            "composite depth and is presented as provisional.",
            size=12.5, color=MUTED, spacing=1.3)

    # 11 Conclusions
    s = add_slide(prs)
    header(s, "Conclusions", "What to use, and what to watch")
    textbox(s, Inches(0.9), Inches(2.0), Inches(11.5), Inches(5),
            "Milan  -  use S2_stack (CLMS labels). Wins on all three techniques; its edge over "
            "S2_median and emb_RF is statistically confirmed, its edge over S2_percentile is not.\n\n"
            "Hanoi  -  use emb_localrf. Locally retrained embedding features win cleanly; do not "
            "deploy the zero-shot embedding model.\n\n"
            "HCMC  -  s2_median_zeroshot performs best, but provisionally: the composite is thin "
            "and cannot be checked against a percentile variant.\n\n"
            "Training label  -  do not train on GHSL where a CLMS-quality reference exists; the "
            "GHSL label's bias propagates into the map.\n\n"
            "Benchmarks  -  CLMS is competitive on continuous error in Milan; GHSL is the weakest "
            "benchmark everywhere, driven by the built-up vs imperviousness mismatch.",
            size=13, spacing=1.4)

    prs.save(OUT_PPTX)
    print("saved:", OUT_PPTX)
    print("figures in:", os.path.abspath(FIGS))


if __name__ == "__main__":
    build()